# Dataclasses como DTO (SOLUCIONES): conversión entre CSV, JSON y XML

**Módulo:** Acceso a Datos · **Tema 1:** Manejo de ficheros

> Este cuaderno se ha elaborado con ayuda de inteligencia artificial y ha sido revisado por el profesor.

## El objetivo

Un **DTO** (*Data Transfer Object*) es un objeto sin lógica de negocio cuya función es **transportar datos**. En este cuaderno lo usaremos como **paso intermedio** para convertir ficheros entre formatos:

```
             lector                         escritor
 libros.csv ────────┐                    ┌──────────▶ libros.json
 libros.json ───────┼──▶ list[Libro] ────┼──────────▶ libros.xml
 libros.xml ────────┘      (DTO)         └──────────▶ libros.csv
```

Cada formato solo necesita **dos funciones**: una que lo lee y devuelve una lista de DTO, y otra que recibe esa lista y la escribe. El resto del programa trabaja siempre con objetos `Libro`, nunca con filas de CSV, diccionarios de JSON o elementos de XML.

Con 3 formatos bastan 6 funciones. Si convirtiéramos directamente de un formato a otro harían falta 6 conversores distintos, y con 5 formatos ya serían 20 frente a 10.

## Qué vamos a usar

Solo el decorador `@dataclass` y las utilidades de su módulo:

| Herramienta | Para qué |
|---|---|
| `@dataclass` | Genera `__init__`, `__repr__` y `__eq__` |
| `campo: tipo = valor` | Campos opcionales con valor por defecto |
| `field(default_factory=list)` | Valores por defecto mutables |
| `field(init=False)` | Campos calculados que no se pasan al constructor |
| `__post_init__` | Validar y calcular campos al crear el objeto |
| `asdict(obj)` | Convertir el DTO en diccionario (recursivo) |
| `fields(clase)` | Recorrer los campos de forma genérica |
| `replace(obj, ...)` | Crear una copia con campos cambiados |
| `@dataclass(frozen=True)` | DTO inmutables |

## Cómo trabajar

Cada ejercicio tiene una celda para tu código y, a continuación, una celda de **comprobación** que no debes modificar. Si todo está bien, imprime `✔`. Si algo falla, el mensaje del `AssertionError` te dice qué revisar.

Ejecuta las celdas **en orden**: los ejercicios se apoyan en los anteriores. La dificultad se indica con ★, ★★ o ★★★.

## Preparación

Esta celda importa lo necesario y crea en la carpeta `datos/` los ficheros de entrada. Las salidas se guardan en `salida/`. Ejecútala antes de empezar.

In [ ]:
import csv
import json
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field, fields, asdict, replace, FrozenInstanceError
from datetime import date
from pathlib import Path

DATOS = Path("datos")
SALIDA = Path("salida")
DATOS.mkdir(exist_ok=True)
SALIDA.mkdir(exist_ok=True)

# Datos esperados tras leer cualquiera de los tres ficheros (los usan las comprobaciones)
ESPERADOS = [
    {"isbn": "9788420412146", "titulo": "Don Quijote de la Mancha", "autor": "Miguel de Cervantes",
     "anio": 1605, "precio": 22.5, "disponible": True, "editorial": "Alfaguara",
     "etiquetas": ["clásico", "novela"]},
    {"isbn": "9788437604947", "titulo": "Cien años de soledad", "autor": "Gabriel García Márquez",
     "anio": 1967, "precio": 19.95, "disponible": True, "editorial": "Cátedra",
     "etiquetas": ["novela", "realismo mágico"]},
    {"isbn": "9788466333917", "titulo": "La sombra del viento", "autor": "Carlos Ruiz Zafón",
     "anio": 2001, "precio": 9.95, "disponible": False, "editorial": None,
     "etiquetas": ["novela", "misterio"]},
    {"isbn": "9788490663196", "titulo": "Patria", "autor": "Fernando Aramburu",
     "anio": 2016, "precio": 21.9, "disponible": True, "editorial": "Tusquets",
     "etiquetas": []},
]

(DATOS / "libros.csv").write_text(
    "isbn,titulo,autor,anio,precio,disponible,editorial,etiquetas\n"
    "9788420412146,Don Quijote de la Mancha,Miguel de Cervantes,1605,22.50,sí,Alfaguara,clásico|novela\n"
    "9788437604947,Cien años de soledad,Gabriel García Márquez,1967,19.95,sí,Cátedra,novela|realismo mágico\n"
    "9788466333917,La sombra del viento,Carlos Ruiz Zafón,2001,9.95,no,,novela|misterio\n"
    "9788490663196,Patria,Fernando Aramburu,2016,21.90,sí,Tusquets,\n",
    encoding="utf-8",
)

(DATOS / "libros.json").write_text(json.dumps(ESPERADOS, ensure_ascii=False, indent=2), encoding="utf-8")

(DATOS / "libros.xml").write_text("""<?xml version="1.0" encoding="UTF-8"?>
<libros>
  <libro isbn="9788420412146" disponible="true">
    <titulo>Don Quijote de la Mancha</titulo>
    <autor>Miguel de Cervantes</autor>
    <anio>1605</anio>
    <precio>22.50</precio>
    <editorial>Alfaguara</editorial>
    <etiquetas>
      <etiqueta>clásico</etiqueta>
      <etiqueta>novela</etiqueta>
    </etiquetas>
  </libro>
  <libro isbn="9788437604947" disponible="true">
    <titulo>Cien años de soledad</titulo>
    <autor>Gabriel García Márquez</autor>
    <anio>1967</anio>
    <precio>19.95</precio>
    <editorial>Cátedra</editorial>
    <etiquetas>
      <etiqueta>novela</etiqueta>
      <etiqueta>realismo mágico</etiqueta>
    </etiquetas>
  </libro>
  <libro isbn="9788466333917" disponible="false">
    <titulo>La sombra del viento</titulo>
    <autor>Carlos Ruiz Zafón</autor>
    <anio>2001</anio>
    <precio>9.95</precio>
    <etiquetas>
      <etiqueta>novela</etiqueta>
      <etiqueta>misterio</etiqueta>
    </etiquetas>
  </libro>
  <libro isbn="9788490663196" disponible="true">
    <titulo>Patria</titulo>
    <autor>Fernando Aramburu</autor>
    <anio>2016</anio>
    <precio>21.90</precio>
    <editorial>Tusquets</editorial>
    <etiquetas/>
  </libro>
</libros>
""", encoding="utf-8")

(DATOS / "pedido.json").write_text("""{
  "numero": 1001,
  "fecha": "2026-10-20",
  "cliente": {"nombre": "Ana Martín", "email": "ana.martin@example.com"},
  "lineas": [
    {"isbn": "9788420412146", "cantidad": 2, "precio_unitario": 22.5},
    {"isbn": "9788466333917", "cantidad": 1, "precio_unitario": 9.95},
    {"isbn": "9788490663196", "cantidad": 3, "precio_unitario": 21.9}
  ]
}
""", encoding="utf-8")

for fichero in sorted(DATOS.iterdir()):
    print(fichero)

Echa un vistazo a cómo representa cada formato los mismos datos. Fíjate sobre todo en tres diferencias que tendrás que resolver al leer:

- **Tipos.** En CSV y XML todo es texto: `"1605"`, `"22.50"`. En JSON los números y los booleanos ya llegan con su tipo.
- **Valores ausentes.** *La sombra del viento* no tiene editorial: en CSV es una celda vacía, en JSON es `null` y en XML falta el elemento `<editorial>`.
- **Listas.** Las etiquetas son una cadena separada por `|` en CSV, un array en JSON y elementos hijos en XML.

In [ ]:
print((DATOS / "libros.csv").read_text(encoding="utf-8"))
print((DATOS / "libros.json").read_text(encoding="utf-8")[:420], "...")

---
## Ejercicio 1 ★ · Tu primera dataclass

Define la clase `Libro` con estos campos, **en este orden** y con estas anotaciones de tipo:

| Campo | Tipo |
|---|---|
| `isbn` | `str` |
| `titulo` | `str` |
| `autor` | `str` |
| `anio` | `int` |
| `precio` | `float` |

No escribas ningún método: el decorador `@dataclass` los genera.

In [ ]:
@dataclass
class Libro:
    isbn: str
    titulo: str
    autor: str
    anio: int
    precio: float

In [ ]:
# Comprobación del ejercicio 1 (no la modifiques)
assert [f.name for f in fields(Libro)] == ["isbn", "titulo", "autor", "anio", "precio"], \
    "Los nombres de los campos o su orden no coinciden con la tabla"
assert [f.type for f in fields(Libro)] == [str, str, str, int, float], "Revisa las anotaciones de tipo"

a = Libro("9788420412146", "Don Quijote de la Mancha", "Miguel de Cervantes", 1605, 22.5)
b = Libro(isbn="9788420412146", titulo="Don Quijote de la Mancha", autor="Miguel de Cervantes",
          anio=1605, precio=22.5)
assert a == b and a is not b, "Dos libros con los mismos datos deben ser iguales (==) aunque sean objetos distintos"
assert repr(a) == ("Libro(isbn='9788420412146', titulo='Don Quijote de la Mancha', "
                   "autor='Miguel de Cervantes', anio=1605, precio=22.5)")
print("✔ Ejercicio 1 correcto")

**Para pensar (respuesta).**

- `print(a)` usa `__repr__`, porque la clase no define `__str__`.
- `a == b` usa `__eq__`, que compara los campos como si fueran tuplas: es `True`.
- `a is b` compara identidad y no usa ningún método generado: es `False`, son dos objetos distintos.
- `{a}` lanza `TypeError: unhashable type: 'Libro'`. Al generar `__eq__` en una clase mutable, `@dataclass` pone `__hash__ = None`, porque si el objeto cambiara después de meterlo en un conjunto, su hash dejaría de ser válido. En el ejercicio 11 veremos cómo cambia esto con `frozen=True`.

---
## Ejercicio 2 ★ · Campos opcionales y valores por defecto

Vuelve a definir `Libro` (copia la del ejercicio 1) y añade al final tres campos opcionales:

| Campo | Tipo | Valor si no se indica |
|---|---|---|
| `disponible` | `bool` | `True` |
| `editorial` | `str` o `None` | `None` |
| `etiquetas` | lista de `str` | una lista vacía **nueva** para cada libro |

Pistas:
- Recuerda que los campos con valor por defecto deben ir **después** de los obligatorios.
- `etiquetas: list[str] = []` no funciona. ¿Qué te dice el error? ¿Cómo se resuelve?

In [ ]:
@dataclass
class Libro:
    isbn: str
    titulo: str
    autor: str
    anio: int
    precio: float
    disponible: bool = True
    editorial: str | None = None
    etiquetas: list[str] = field(default_factory=list)

In [ ]:
# Comprobación del ejercicio 2 (no la modifiques)
nombres = [f.name for f in fields(Libro)]
assert nombres == ["isbn", "titulo", "autor", "anio", "precio", "disponible", "editorial", "etiquetas"], \
    f"Campos encontrados: {nombres}"
tipos = {f.name: f.type for f in fields(Libro)}
assert tipos["disponible"] == bool, "disponible debe ser bool"
assert tipos["editorial"] == str | None, "editorial debe anotarse como str | None"
assert tipos["etiquetas"] == list[str], "etiquetas debe anotarse como list[str]"

x = Libro("9788490663196", "Patria", "Fernando Aramburu", 2016, 21.9)
y = Libro("9788437604947", "Cien años de soledad", "Gabriel García Márquez", 1967, 19.95)
assert x.disponible is True and x.editorial is None and x.etiquetas == [], "Revisa los valores por defecto"
x.etiquetas.append("novela")
assert y.etiquetas == [], "Los libros comparten la misma lista de etiquetas: usa field(default_factory=list)"
print("✔ Ejercicio 2 correcto")

---
## Ejercicio 3 ★★ · De CSV a DTO

Escribe la función `leer_csv(ruta)`, que lee `libros.csv` y devuelve una **lista de `Libro`**.

`csv.DictReader` devuelve cada fila como un diccionario **de cadenas**, así que tienes que convertir cada valor al tipo del campo:

| Columna | En el CSV | En el `Libro` |
|---|---|---|
| `anio`, `precio` | `"1605"`, `"22.50"` | `int`, `float` |
| `disponible` | `"sí"` / `"no"` | `True` / `False` |
| `editorial` | celda vacía `""` | `None` |
| `etiquetas` | `"clásico\|novela"` o `""` | `["clásico", "novela"]` o `[]` |

Cuidado: `"".split("|")` devuelve `[""]`, no `[]`.

In [ ]:
def leer_csv(ruta: Path) -> list[Libro]:
    libros = []
    with open(ruta, encoding="utf-8", newline="") as f:
        for fila in csv.DictReader(f):
            libros.append(Libro(
                isbn=fila["isbn"],
                titulo=fila["titulo"],
                autor=fila["autor"],
                anio=int(fila["anio"]),
                precio=float(fila["precio"]),
                disponible=fila["disponible"] == "sí",
                editorial=fila["editorial"] or None,
                etiquetas=fila["etiquetas"].split("|") if fila["etiquetas"] else [],
            ))
    return libros


libros = leer_csv(DATOS / "libros.csv")
for libro in libros:
    print(libro)

In [ ]:
# Comprobación del ejercicio 3 (no la modifiques)
libros = leer_csv(DATOS / "libros.csv")
assert isinstance(libros, list) and len(libros) == 4, "Debe devolver una lista con 4 libros"
assert all(isinstance(l, Libro) for l in libros), "Cada elemento debe ser un Libro, no un diccionario"
for obtenido, esperado in zip(libros, ESPERADOS):
    assert asdict(obtenido) == esperado, f"\nObtenido: {asdict(obtenido)}\nEsperado: {esperado}"
print("✔ Ejercicio 3 correcto")

---
## Ejercicio 4 ★ · DTO ⇄ JSON

Escribe dos funciones:

- `escribir_json(libros, ruta)`: guarda la lista en un fichero JSON. Usa `asdict` para convertir cada libro en diccionario. Pasa `ensure_ascii=False` e `indent=2` a `json.dump` para que el fichero sea legible y conserve las tildes.
- `leer_json(ruta)`: lee el fichero y devuelve una lista de `Libro`.

Aquí la lectura es mucho más sencilla que en CSV: las claves del JSON coinciden con los nombres de los campos y los valores ya tienen el tipo correcto. ¿Cómo puedes crear un `Libro` a partir de un diccionario en una sola expresión?

In [ ]:
def escribir_json(libros: list[Libro], ruta: Path) -> None:
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump([asdict(libro) for libro in libros], f, ensure_ascii=False, indent=2)


def leer_json(ruta: Path) -> list[Libro]:
    with open(ruta, encoding="utf-8") as f:
        return [Libro(**datos) for datos in json.load(f)]

In [ ]:
# Comprobación del ejercicio 4 (no la modifiques)
desde_csv = leer_csv(DATOS / "libros.csv")
desde_json = leer_json(DATOS / "libros.json")
assert all(isinstance(l, Libro) for l in desde_json), "leer_json debe devolver objetos Libro"
assert desde_json == desde_csv, "Los libros leídos del JSON no coinciden con los del CSV"

escribir_json(desde_csv, SALIDA / "libros.json")
texto = (SALIDA / "libros.json").read_text(encoding="utf-8")
assert "García" in texto, "Las tildes aparecen escapadas (\\u00ed...): usa ensure_ascii=False"
assert leer_json(SALIDA / "libros.json") == desde_csv, "Al releer el JSON escrito no se obtienen los mismos libros"
print("✔ Ejercicio 4 correcto")

---
## Ejercicio 5 ★★ · De XML a DTO

Escribe `leer_xml(ruta)` con `xml.etree.ElementTree`. Recuerda la estructura del fichero:

```xml
<libro isbn="9788420412146" disponible="true">
  <titulo>Don Quijote de la Mancha</titulo>
  ...
  <editorial>Alfaguara</editorial>      <!-- puede no existir -->
  <etiquetas>
    <etiqueta>clásico</etiqueta>
    <etiqueta>novela</etiqueta>
  </etiquetas>
</libro>
```

Pistas:
- `isbn` y `disponible` son **atributos**: `elemento.get("isbn")`.
- `elemento.findtext("titulo")` devuelve el texto del hijo, o `None` si el hijo no existe. Justo lo que necesitas para `editorial`.
- Para las etiquetas, `findall` admite rutas: `elemento.findall("etiquetas/etiqueta")`.

In [ ]:
def leer_xml(ruta: Path) -> list[Libro]:
    raiz = ET.parse(ruta).getroot()
    libros = []
    for elemento in raiz.findall("libro"):
        libros.append(Libro(
            isbn=elemento.get("isbn"),
            titulo=elemento.findtext("titulo"),
            autor=elemento.findtext("autor"),
            anio=int(elemento.findtext("anio")),
            precio=float(elemento.findtext("precio")),
            disponible=elemento.get("disponible") == "true",
            editorial=elemento.findtext("editorial"),
            etiquetas=[e.text for e in elemento.findall("etiquetas/etiqueta")],
        ))
    return libros

In [ ]:
# Comprobación del ejercicio 5 (no la modifiques)
desde_xml = leer_xml(DATOS / "libros.xml")
assert all(isinstance(l, Libro) for l in desde_xml), "leer_xml debe devolver objetos Libro"
for obtenido, esperado in zip(desde_xml, ESPERADOS):
    assert asdict(obtenido) == esperado, f"\nObtenido: {asdict(obtenido)}\nEsperado: {esperado}"
assert desde_xml == leer_csv(DATOS / "libros.csv")
print("✔ Ejercicio 5 correcto")

Fíjate en lo que has conseguido: tres ficheros con formatos muy distintos producen **exactamente la misma lista de DTO**, y se puede comprobar con un simple `==` gracias al `__eq__` generado.

---
## Ejercicio 6 ★★ · De DTO a XML y a CSV

Escribe las dos funciones de escritura que faltan. Deben producir ficheros con **la misma estructura** que los de `datos/`, de forma que tus propias funciones de lectura puedan volver a leerlos.

**`escribir_xml(libros, ruta)`**
- Crea los elementos con `ET.Element` y `ET.SubElement`. Los atributos se pasan como argumentos por nombre: `ET.SubElement(raiz, "libro", isbn=libro.isbn)`.
- En XML todo es texto: convierte números y booleanos con `str()` o con una expresión condicional.
- Si `editorial` es `None`, no escribas el elemento.
- Antes de guardar, `ET.indent(arbol)` sangra el documento. Guarda con `arbol.write(ruta, encoding="utf-8", xml_declaration=True)`.

**`escribir_csv(libros, ruta)`**
- No escribas a mano la lista de columnas: obtenla con `fields(Libro)`. Así, si mañana añades un campo a `Libro`, el CSV lo incluye solo.
- Parte de `asdict(libro)` y ajusta los tres campos que no se escriben tal cual: `disponible`, `editorial` y `etiquetas`.

In [ ]:
def escribir_xml(libros: list[Libro], ruta: Path) -> None:
    raiz = ET.Element("libros")
    for libro in libros:
        elemento = ET.SubElement(raiz, "libro", isbn=libro.isbn,
                                 disponible="true" if libro.disponible else "false")
        ET.SubElement(elemento, "titulo").text = libro.titulo
        ET.SubElement(elemento, "autor").text = libro.autor
        ET.SubElement(elemento, "anio").text = str(libro.anio)
        ET.SubElement(elemento, "precio").text = str(libro.precio)
        if libro.editorial is not None:
            ET.SubElement(elemento, "editorial").text = libro.editorial
        etiquetas = ET.SubElement(elemento, "etiquetas")
        for etiqueta in libro.etiquetas:
            ET.SubElement(etiquetas, "etiqueta").text = etiqueta
    arbol = ET.ElementTree(raiz)
    ET.indent(arbol)
    arbol.write(ruta, encoding="utf-8", xml_declaration=True)


def escribir_csv(libros: list[Libro], ruta: Path) -> None:
    columnas = [f.name for f in fields(Libro)]
    with open(ruta, "w", encoding="utf-8", newline="") as f:
        escritor = csv.DictWriter(f, fieldnames=columnas)
        escritor.writeheader()
        for libro in libros:
            fila = asdict(libro)
            fila["disponible"] = "sí" if libro.disponible else "no"
            fila["editorial"] = libro.editorial or ""
            fila["etiquetas"] = "|".join(libro.etiquetas)
            escritor.writerow(fila)

In [ ]:
# Comprobación del ejercicio 6 (no la modifiques)
originales = leer_csv(DATOS / "libros.csv")

escribir_xml(originales, SALIDA / "libros.xml")
xml = (SALIDA / "libros.xml").read_text(encoding="utf-8")
assert xml.startswith("<?xml"), "Falta la declaración XML: usa xml_declaration=True"
assert xml.count("<editorial>") == 3, "Solo tres libros tienen editorial: no escribas <editorial> si es None"
assert leer_xml(SALIDA / "libros.xml") == originales, "Al releer el XML escrito no se obtienen los mismos libros"

escribir_csv(originales, SALIDA / "libros.csv")
cabecera = (SALIDA / "libros.csv").read_text(encoding="utf-8").splitlines()[0]
assert cabecera == "isbn,titulo,autor,anio,precio,disponible,editorial,etiquetas", f"Cabecera: {cabecera}"
assert leer_csv(SALIDA / "libros.csv") == originales, "Al releer el CSV escrito no se obtienen los mismos libros"
print("✔ Ejercicio 6 correcto")

---
## Ejercicio 7 ★★ · Validar el DTO con `__post_init__`

Los datos que llegan de un fichero pueden estar mal. Vuelve a definir `Libro` (copia la del ejercicio 2) y añádele un método `__post_init__` que:

1. **Normalice** `titulo` y `autor` quitándoles los espacios de los extremos.
2. **Valide** los datos y lance `ValueError` con un mensaje claro si:
   - `isbn` no tiene exactamente 13 caracteres o no son todos dígitos;
   - `titulo` queda vacío tras normalizarlo;
   - `precio` es negativo;
   - `anio` es mayor que el año actual (`date.today().year`).

Como `__post_init__` se ejecuta al final del `__init__` generado, **todos** los lectores que ya has escrito validan automáticamente sin cambiar ni una línea, porque crean los libros llamando a `Libro(...)`.

In [ ]:
@dataclass
class Libro:
    isbn: str
    titulo: str
    autor: str
    anio: int
    precio: float
    disponible: bool = True
    editorial: str | None = None
    etiquetas: list[str] = field(default_factory=list)

    def __post_init__(self):
        self.titulo = self.titulo.strip()
        self.autor = self.autor.strip()
        if len(self.isbn) != 13 or not self.isbn.isdigit():
            raise ValueError(f"ISBN no válido: {self.isbn!r}")
        if not self.titulo:
            raise ValueError("El título no puede estar vacío")
        if self.precio < 0:
            raise ValueError(f"Precio negativo: {self.precio}")
        if self.anio > date.today().year:
            raise ValueError(f"Año de publicación futuro: {self.anio}")

In [ ]:
# Comprobación del ejercicio 7 (no la modifiques)
BASE = dict(isbn="9788420412146", titulo="Don Quijote", autor="Cervantes", anio=1605, precio=22.5)

def lanza_value_error(**cambios) -> bool:
    try:
        Libro(**(BASE | cambios))
    except ValueError:
        return True
    return False

assert hasattr(Libro, "__post_init__"), "Libro no define __post_init__"
libro = Libro(**(BASE | {"titulo": "  Don Quijote  ", "autor": " Cervantes\n"}))
assert libro.titulo == "Don Quijote" and libro.autor == "Cervantes", "titulo y autor deben quedar sin espacios"
assert lanza_value_error(isbn="978842041214"), "Un ISBN de 12 caracteres debe lanzar ValueError"
assert lanza_value_error(isbn="978842041214X"), "Un ISBN con letras debe lanzar ValueError"
assert lanza_value_error(titulo="   "), "Un título vacío debe lanzar ValueError"
assert lanza_value_error(precio=-1.0), "Un precio negativo debe lanzar ValueError"
assert lanza_value_error(anio=date.today().year + 1), "Un año futuro debe lanzar ValueError"
assert not lanza_value_error(precio=0.0), "Un precio de 0 es válido"

# replace() crea el objeto llamando a __init__, así que también valida
try:
    replace(libro, precio=-5)
    assert False, "replace() con un precio negativo debería lanzar ValueError"
except ValueError:
    pass

# Los lectores de los ejercicios anteriores usan la nueva clase sin cambios
assert len(leer_csv(DATOS / "libros.csv")) == 4 and len(leer_xml(DATOS / "libros.xml")) == 4
print("✔ Ejercicio 7 correcto")

---
## Ejercicio 8 ★★★ · DTO anidados y campos calculados

Un DTO puede contener otros DTO. Vamos a representar un **pedido** de la librería (`datos/pedido.json`):

```json
{
  "numero": 1001,
  "fecha": "2026-10-20",
  "cliente": {"nombre": "Ana Martín", "email": "ana.martin@example.com"},
  "lineas": [
    {"isbn": "9788420412146", "cantidad": 2, "precio_unitario": 22.5},
    ...
  ]
}
```

**a)** Define tres dataclasses:

- `Cliente`: `nombre` (`str`) y `email` (`str`).
- `LineaPedido`: `isbn` (`str`), `cantidad` (`int`), `precio_unitario` (`float`) y `subtotal` (`float`). El `subtotal` **no se pasa al constructor**: se calcula como `cantidad × precio_unitario`, redondeado a 2 decimales.
- `Pedido`: `numero` (`int`), `fecha` (`date`), `cliente` (`Cliente`), `lineas` (lista de `LineaPedido`, vacía por defecto) y `total` (`float`). El `total` tampoco se pasa: es la suma de los subtotales, redondeada a 2 decimales.

**b)** Escribe dos funciones de conversión:

- `pedido_desde_dict(datos)`: crea el `Pedido` a partir del diccionario. Convierte la fecha con `date.fromisoformat` y construye los objetos anidados.
- `pedido_a_dict(pedido)`: hace lo contrario. `asdict` ya convierte los DTO anidados, pero deja la fecha como objeto `date`, que `json.dump` no sabe escribir: pásala a texto con `.isoformat()`.

**c)** Con ellas, escribe `leer_pedido_json(ruta)` y `escribir_pedido_json(pedido, ruta)`.

Ten en cuenta que `asdict` incluye también `subtotal` y `total`, así que el JSON que escribas los tendrá. Al leerlo, `LineaPedido(**datos)` fallaría, porque esos campos no son argumentos del constructor. ¿Cómo lo resuelves en `pedido_desde_dict`?

In [ ]:
@dataclass
class Cliente:
    nombre: str
    email: str


@dataclass
class LineaPedido:
    isbn: str
    cantidad: int
    precio_unitario: float
    subtotal: float = field(init=False)

    def __post_init__(self):
        self.subtotal = round(self.cantidad * self.precio_unitario, 2)


@dataclass
class Pedido:
    numero: int
    fecha: date
    cliente: Cliente
    lineas: list[LineaPedido] = field(default_factory=list)
    total: float = field(init=False)

    def __post_init__(self):
        self.total = round(sum(linea.subtotal for linea in self.lineas), 2)


def pedido_desde_dict(datos: dict) -> Pedido:
    return Pedido(
        numero=datos["numero"],
        fecha=date.fromisoformat(datos["fecha"]),
        cliente=Cliente(**datos["cliente"]),
        # No se usa LineaPedido(**l) porque el diccionario puede traer "subtotal"
        lineas=[
            LineaPedido(isbn=l["isbn"], cantidad=l["cantidad"], precio_unitario=l["precio_unitario"])
            for l in datos["lineas"]
        ],
    )


def pedido_a_dict(pedido: Pedido) -> dict:
    datos = asdict(pedido)
    datos["fecha"] = pedido.fecha.isoformat()
    return datos


def leer_pedido_json(ruta: Path) -> Pedido:
    with open(ruta, encoding="utf-8") as f:
        return pedido_desde_dict(json.load(f))


def escribir_pedido_json(pedido: Pedido, ruta: Path) -> None:
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(pedido_a_dict(pedido), f, ensure_ascii=False, indent=2)

In [ ]:
# Comprobación del ejercicio 8 (no la modifiques)
assert [f.name for f in fields(Cliente)] == ["nombre", "email"]
assert [f.name for f in fields(LineaPedido)] == ["isbn", "cantidad", "precio_unitario", "subtotal"]
assert [f.name for f in fields(Pedido)] == ["numero", "fecha", "cliente", "lineas", "total"]

linea = LineaPedido("9788490663196", 3, 21.9)
assert linea.subtotal == 65.7, f"subtotal = {linea.subtotal}; ¿has redondeado a 2 decimales?"
try:
    LineaPedido("9788490663196", 3, 21.9, 65.7)
    assert False, "subtotal no debe poder pasarse al constructor: usa field(init=False)"
except TypeError:
    pass
assert Pedido(1, date(2026, 1, 1), Cliente("x", "y")).lineas == [], "lineas debe ser una lista vacía por defecto"

pedido = leer_pedido_json(DATOS / "pedido.json")
assert isinstance(pedido, Pedido) and isinstance(pedido.cliente, Cliente), "El cliente debe ser un objeto Cliente"
assert all(isinstance(l, LineaPedido) for l in pedido.lineas), "Las líneas deben ser objetos LineaPedido"
assert pedido.fecha == date(2026, 10, 20), "La fecha debe ser un objeto date"
assert pedido.total == 120.65, f"total = {pedido.total}"

escribir_pedido_json(pedido, SALIDA / "pedido.json")
assert '"fecha": "2026-10-20"' in (SALIDA / "pedido.json").read_text(encoding="utf-8")
assert leer_pedido_json(SALIDA / "pedido.json") == pedido, "Al releer el JSON escrito no se obtiene el mismo pedido"
print(pedido)
print("✔ Ejercicio 8 correcto")

---
## Ejercicio 9 ★★★ · El pedido en XML y en CSV

Ahora escribe el mismo `Pedido` en los otros dos formatos y vuelve a leerlo.

**a) XML.** Usa esta estructura. Los datos derivados (`subtotal` y `total`) **no se guardan**: se recalculan al crear los objetos.

```xml
<pedido numero="1001" fecha="2026-10-20">
  <cliente>
    <nombre>Ana Martín</nombre>
    <email>ana.martin@example.com</email>
  </cliente>
  <lineas>
    <linea isbn="9788420412146" cantidad="2" precio_unitario="22.5" />
    ...
  </lineas>
</pedido>
```

**b) CSV.** Un CSV es una tabla plana: no admite objetos anidados. La solución habitual es **desnormalizar**: una fila por línea de pedido, repitiendo en cada fila los datos del pedido y del cliente. Usa estas columnas:

```
numero,fecha,cliente_nombre,cliente_email,isbn,cantidad,precio_unitario,subtotal
```

Para leerlo, toma los datos del pedido y del cliente de la primera fila y construye una `LineaPedido` por cada fila. Puedes suponer que el fichero contiene un único pedido.

In [ ]:
def escribir_pedido_xml(pedido: Pedido, ruta: Path) -> None:
    raiz = ET.Element("pedido", numero=str(pedido.numero), fecha=pedido.fecha.isoformat())
    cliente = ET.SubElement(raiz, "cliente")
    ET.SubElement(cliente, "nombre").text = pedido.cliente.nombre
    ET.SubElement(cliente, "email").text = pedido.cliente.email
    lineas = ET.SubElement(raiz, "lineas")
    for linea in pedido.lineas:
        ET.SubElement(lineas, "linea", isbn=linea.isbn, cantidad=str(linea.cantidad),
                      precio_unitario=str(linea.precio_unitario))
    arbol = ET.ElementTree(raiz)
    ET.indent(arbol)
    arbol.write(ruta, encoding="utf-8", xml_declaration=True)


def leer_pedido_xml(ruta: Path) -> Pedido:
    raiz = ET.parse(ruta).getroot()
    return Pedido(
        numero=int(raiz.get("numero")),
        fecha=date.fromisoformat(raiz.get("fecha")),
        cliente=Cliente(nombre=raiz.findtext("cliente/nombre"), email=raiz.findtext("cliente/email")),
        lineas=[
            LineaPedido(isbn=e.get("isbn"), cantidad=int(e.get("cantidad")),
                        precio_unitario=float(e.get("precio_unitario")))
            for e in raiz.findall("lineas/linea")
        ],
    )


COLUMNAS_PEDIDO = ["numero", "fecha", "cliente_nombre", "cliente_email",
                   "isbn", "cantidad", "precio_unitario", "subtotal"]


def escribir_pedido_csv(pedido: Pedido, ruta: Path) -> None:
    with open(ruta, "w", encoding="utf-8", newline="") as f:
        escritor = csv.DictWriter(f, fieldnames=COLUMNAS_PEDIDO)
        escritor.writeheader()
        for linea in pedido.lineas:
            escritor.writerow({
                "numero": pedido.numero,
                "fecha": pedido.fecha.isoformat(),
                "cliente_nombre": pedido.cliente.nombre,
                "cliente_email": pedido.cliente.email,
                "isbn": linea.isbn,
                "cantidad": linea.cantidad,
                "precio_unitario": linea.precio_unitario,
                "subtotal": linea.subtotal,
            })


def leer_pedido_csv(ruta: Path) -> Pedido:
    with open(ruta, encoding="utf-8", newline="") as f:
        filas = list(csv.DictReader(f))
    primera = filas[0]
    return Pedido(
        numero=int(primera["numero"]),
        fecha=date.fromisoformat(primera["fecha"]),
        cliente=Cliente(nombre=primera["cliente_nombre"], email=primera["cliente_email"]),
        lineas=[
            LineaPedido(isbn=fila["isbn"], cantidad=int(fila["cantidad"]),
                        precio_unitario=float(fila["precio_unitario"]))
            for fila in filas
        ],
    )

In [ ]:
# Comprobación del ejercicio 9 (no la modifiques)
pedido = leer_pedido_json(DATOS / "pedido.json")

escribir_pedido_xml(pedido, SALIDA / "pedido.xml")
xml = (SALIDA / "pedido.xml").read_text(encoding="utf-8")
assert "subtotal" not in xml and "total" not in xml, "No guardes los datos derivados en el XML"
assert leer_pedido_xml(SALIDA / "pedido.xml") == pedido, "Al releer el XML escrito no se obtiene el mismo pedido"

escribir_pedido_csv(pedido, SALIDA / "pedido.csv")
lineas_csv = (SALIDA / "pedido.csv").read_text(encoding="utf-8").splitlines()
assert lineas_csv[0] == "numero,fecha,cliente_nombre,cliente_email,isbn,cantidad,precio_unitario,subtotal", \
    f"Cabecera: {lineas_csv[0]}"
assert len(lineas_csv) == 4, "Debe haber una cabecera y una fila por línea de pedido"
assert leer_pedido_csv(SALIDA / "pedido.csv") == pedido, "Al releer el CSV escrito no se obtiene el mismo pedido"
print((SALIDA / "pedido.csv").read_text(encoding="utf-8"))
print("✔ Ejercicio 9 correcto")

---
## Ejercicio 10 ★★ · Un conversor genérico

Con los lectores y escritores de libros que ya tienes, escribe `convertir(origen, destino)`, que convierte un fichero de libros de cualquier formato a cualquier otro según la **extensión** de cada ruta (`ruta.suffix`).

- Guarda las funciones en dos diccionarios, `LECTORES` y `ESCRITORES`, cuyas claves sean `".csv"`, `".json"` y `".xml"`. En Python las funciones son objetos y se pueden guardar en un diccionario y llamar después.
- Si alguna extensión no está admitida, lanza `ValueError`.
- La función devuelve el número de libros convertidos.

Fíjate en que `convertir` no sabe nada de ningún formato: solo conecta un lector con un escritor a través de la lista de DTO.

In [ ]:
LECTORES = {".csv": leer_csv, ".json": leer_json, ".xml": leer_xml}
ESCRITORES = {".csv": escribir_csv, ".json": escribir_json, ".xml": escribir_xml}


def convertir(origen: Path, destino: Path) -> int:
    lector = LECTORES.get(origen.suffix.lower())
    escritor = ESCRITORES.get(destino.suffix.lower())
    if lector is None:
        raise ValueError(f"Formato de origen no admitido: {origen.suffix!r}")
    if escritor is None:
        raise ValueError(f"Formato de destino no admitido: {destino.suffix!r}")
    libros = lector(origen)
    escritor(libros, destino)
    return len(libros)

In [ ]:
# Comprobación del ejercicio 10 (no la modifiques)
assert set(LECTORES) == set(ESCRITORES) == {".csv", ".json", ".xml"}, "Faltan formatos en los diccionarios"

# Viaje completo: CSV -> JSON -> XML -> CSV
assert convertir(DATOS / "libros.csv", SALIDA / "viaje.json") == 4
assert convertir(SALIDA / "viaje.json", SALIDA / "viaje.xml") == 4
assert convertir(SALIDA / "viaje.xml", SALIDA / "viaje.csv") == 4
assert leer_csv(SALIDA / "viaje.csv") == leer_csv(DATOS / "libros.csv"), \
    "Tras el viaje CSV → JSON → XML → CSV los libros deberían ser los mismos"

for origen, destino in [("libros.txt", "x.json"), ("libros.csv", "x.yaml")]:
    try:
        convertir(DATOS / origen, SALIDA / destino)
        assert False, f"convertir({origen!r}, {destino!r}) debería lanzar ValueError"
    except ValueError:
        pass
print("✔ Ejercicio 10 correcto")

---
## Ejercicio 11 ★★ · DTO inmutables con `frozen=True`

Un DTO se crea, viaja y se lee, pero normalmente **no debería modificarse** por el camino. Con `@dataclass(frozen=True)` cualquier asignación a un campo lanza `FrozenInstanceError`.

Imagina que el catálogo web solo necesita el ISBN, el título y el precio de cada libro. Se suele crear un DTO específico con **solo** los datos que necesita el destino.

1. Define `LibroCatalogo`, inmutable, con `isbn` (`str`), `titulo` (`str`) y `precio` (`float`).
2. Escribe `a_catalogo(libros)`, que convierte una lista de `Libro` en una lista de `LibroCatalogo`.
3. Escribe `aplicar_descuento(catalogo, porcentaje)`, que devuelve una **lista nueva** con los precios rebajados y redondeados a 2 decimales. Como no puedes modificar los objetos, usa `replace()` para crear copias.

In [ ]:
@dataclass(frozen=True)
class LibroCatalogo:
    isbn: str
    titulo: str
    precio: float


def a_catalogo(libros: list[Libro]) -> list[LibroCatalogo]:
    return [LibroCatalogo(isbn=l.isbn, titulo=l.titulo, precio=l.precio) for l in libros]


def aplicar_descuento(catalogo: list[LibroCatalogo], porcentaje: float) -> list[LibroCatalogo]:
    factor = 1 - porcentaje / 100
    return [replace(l, precio=round(l.precio * factor, 2)) for l in catalogo]

In [ ]:
# Comprobación del ejercicio 11 (no la modifiques)
assert [f.name for f in fields(LibroCatalogo)] == ["isbn", "titulo", "precio"]
catalogo = a_catalogo(leer_csv(DATOS / "libros.csv"))
assert all(isinstance(l, LibroCatalogo) for l in catalogo) and len(catalogo) == 4

try:
    catalogo[0].precio = 0
    assert False, "LibroCatalogo debería ser inmutable: usa @dataclass(frozen=True)"
except FrozenInstanceError:
    pass

rebajados = aplicar_descuento(catalogo, 10)
assert [l.precio for l in rebajados] == [20.25, 17.95, 8.96, 19.71], f"Precios: {[l.precio for l in rebajados]}"
assert [l.precio for l in catalogo] == [22.5, 19.95, 9.95, 21.9], "El catálogo original no debe cambiar"

# Al ser inmutable (y con campos inmutables), ahora sí es hashable
assert len(set(catalogo + catalogo)) == 4, "LibroCatalogo debería poder usarse en un conjunto"
print("✔ Ejercicio 11 correcto")

**Para pensar (respuesta).** Con `frozen=True` (y `eq=True`, que es el valor por defecto), `@dataclass` genera `__hash__` a partir de los campos, porque el objeto ya no puede cambiar. Por eso `set(catalogo)` funciona.

Si `Libro` fuera `frozen=True`, crearlo y asignarle campos fallaría igual, pero calcular su hash lanzaría `TypeError: unhashable type: 'list'`, porque el hash se calcula sobre la tupla de todos los campos y `etiquetas` es una lista. `frozen` impide reasignar el campo (`libro.etiquetas = [...]`), pero no impide modificar la lista que contiene (`libro.etiquetas.append(...)`). Para un DTO totalmente inmutable habría que usar una tupla: `etiquetas: tuple[str, ...] = ()`.

---
## Resumen

| Situación | Solución con `@dataclass` |
|---|---|
| Campo obligatorio | `campo: tipo` |
| Campo opcional | `campo: tipo = valor` (después de los obligatorios) |
| Campo que puede no tener valor | `campo: str \| None = None` |
| Lista o diccionario por defecto | `field(default_factory=list)` |
| Normalizar y validar al crear | `__post_init__` |
| Campo calculado | `field(init=False)` + `__post_init__` |
| DTO → `dict` (para JSON o CSV) | `asdict(obj)` |
| `dict` → DTO | `Clase(**datos)` si las claves coinciden; si no, a mano |
| Columnas de un CSV | `[f.name for f in fields(Clase)]` |
| Copia con cambios | `replace(obj, campo=valor)` |
| DTO que no se puede modificar | `@dataclass(frozen=True)` |

**La idea clave:** cada formato solo necesita un lector y un escritor que traduzcan entre el fichero y la lista de DTO. Todo lo demás (validar, calcular, filtrar, convertir) se hace una sola vez, sobre objetos con tipos, sin importar de dónde vengan los datos ni a dónde vayan.